# Refresh Download of Reviews

In [1]:
import requests
import pandas as pd


# Get top 100 games by player count from SteamSpy
response = requests.get('https://steamspy.com/api.php?request=top100in2weeks')
top_games = pd.DataFrame(response.json()).T #shortand for JSON transpose rows to cols
top_games = top_games.head(100) #get first 100 rows
top_games['appid'] = top_games['appid'].astype(int) #convert appid to integer
top_games[['appid', 'name']].head() #get first 5 rows of appid and name

,appid,name
730,730,Counter-Strike: Global Offensive
1172470,1172470,Apex Legends
578080,578080,PUBG: BATTLEGROUNDS
1623730,1623730,Palworld
440,440,Team Fortress 2


In [2]:
!pip install -Uqq steam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.1/644.1 kB 12.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
import os
from steam.webapi import WebAPI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
STEAM_API_KEY = user_secrets.get_secret("STEAM_API_KEY")

api = WebAPI(key=STEAM_API_KEY)

# Example: Get reviews for a single app
def get_reviews(appid, num_reviews=100, cursor='*'):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        'json': 1,
        'num_per_page': num_reviews,
        'cursor': cursor,
        'filter': 'recent',
        'language': 'english'
    }
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        if 'reviews' in data:
            return { 'cursor': data['cursor'], 'reviews': data['reviews'] }
    return []

In [4]:
# Preview of CS review JSON data
cs_reviews = get_reviews(730, num_reviews=1)
cs_reviews['reviews'][0]

{'recommendationid': '199787535',
 'author': {'steamid': '76561198874365921',
  'num_games_owned': 0,
  'num_reviews': 1,
  'playtime_forever': 5985,
  'playtime_last_two_weeks': 2216,
  'playtime_at_review': 5985,
  'last_played': 1752504068},
 'language': 'english',
 'review': 'I LOVEIT',
 'timestamp_created': 1752504033,
 'timestamp_updated': 1752504033,
 'voted_up': True,
 'votes_up': 0,
 'votes_funny': 0,
 'weighted_vote_score': 0.5,
 'comment_count': 0,
 'steam_purchase': True,
 'received_for_free': False,
 'written_during_early_access': False,
 'primarily_steam_deck': False}

In [5]:
from tqdm import tqdm

all_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    appid = row['appid']
    name = row['name']
    response = get_reviews(appid, num_reviews=10)
    for review in response['reviews']:
        all_reviews.append({
            'appid': appid,
            'name': name,
            'review': review['review'],
            'timestamp_created': review['timestamp_created'],
            'voted_up': review['voted_up'],
            'votes_up': review['votes_up'],
            'votes_funny': review['votes_funny'],
            'weighted_vote_score': review['weighted_vote_score'],
        })

reviews_df = pd.DataFrame(all_reviews)
reviews_df.head()

100%|██████████| 100/100 [00:18<00:00,  5.46it/s]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,I LOVEIT,1752504033,True,0,0,0.5
1,730,Counter-Strike: Global Offensive,kokotina\r\n,1752504004,False,0,0,0.5
2,730,Counter-Strike: Global Offensive,good,1752503928,True,1,0,0.523809552192687988
3,730,Counter-Strike: Global Offensive,Нзх,1752503292,True,0,0,0.5
4,730,Counter-Strike: Global Offensive,game rat hay toi rat thích\r\n,1752503244,True,0,0,0.5


In [6]:
reviews_df.to_csv('reviews.csv', index=False)

# Start Here for analysis

In [7]:
import pandas as pd

df = pd.read_csv('reviews.csv')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   appid                1000 non-null   int64  
 1   name                 1000 non-null   object 
 2   review               994 non-null    object 
 3   timestamp_created    1000 non-null   int64  
 4   voted_up             1000 non-null   bool   
 5   votes_up             1000 non-null   int64  
 6   votes_funny          1000 non-null   int64  
 7   weighted_vote_score  1000 non-null   float64
dtypes: bool(1), float64(1), int64(4), object(2)
memory usage: 55.8+ KB


appid: The unique steam id for each game   
name: The unique game name  
review: The corpus of all text in the review  
timestamp_created: Unix time (epoch) in UTC (POSIX TIME) seconds since 01/01/1970  
`voted_up`: The reviewer's thumb up (True) or thumb down (False) Score `(our target)`  
votes_up: Number of people who upvoted the review  
votes_funny: Number of people who thought the vote was funny  
weighted_vote_score: Steam's helpfullness score - between 0 to 1 - where low scores are likely spam  


In [9]:
df.weighted_vote_score.describe()

count    1000.000000
mean        0.501162
std         0.021378
min         0.254542
25%         0.500000
50%         0.500000
75%         0.500000
max         0.769796
Name: weighted_vote_score, dtype: float64

Looking at the weighted score - we can see the first 75% essentially stay at or below the 50% probability of spam. Why don't we only grab reviews that are substantial by filtering the df to where the score is above 0.50.

In [10]:
df_filtered = df[df.weighted_vote_score > 0.6]
len(df_filtered)

6

Ok, that took us from almost 10k results down to 1571.. we probably should request around 100 with this criteria before filtering - but let's observed some of the comments in the filtered. 

We checked at >= 0.5 and that results in 8771 remaining.. so quite a few right at 0.5 are removed as expected.

Also at >0.6 results in only 104

In [11]:
df_filtered.head()

,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
409,438100,VRChat,Got Banned faster than a pedophile in a playgr...,1487689869,False,27,9,0.633370
660,755790,Ring of Elysium,Was good before they took the snowboards out,1739149789,False,27,0,0.750081
691,550650,Black Squad,My patience is wearing thin. Valofe is going t...,1608410178,False,37,0,0.769796
777,433850,Z1 Battle Royale,Ah the good ole days where lobbies were filled...,1750635523,True,13,3,0.686610
779,433850,Z1 Battle Royale,good times miss this game,1750345191,True,10,0,0.639541


In [12]:
# Pre-filter Reviews
quality_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    n_quality = 0
    appid = row['appid']
    name = row['name']
    cursor = '*' # Initial cursor
    
    while n_quality < 90:
        reviews_data = get_reviews(appid, num_reviews=90, cursor=cursor)
        reviews = reviews_data['reviews']
        cursor = reviews_data.get('cursor') # Update's cursor value

        if not reviews:
            break # early escape if no more reviews to fetch
        
        for review in reviews:
            if float(review['weighted_vote_score']) >= 0.51:
                n_quality += 1
                all_reviews.append({
                    'appid': appid,
                    'name': name,
                    'review': review['review'],
                    'timestamp_created': review['timestamp_created'],
                    'voted_up': review['voted_up'],
                    'votes_up': review['votes_up'],
                    'votes_funny': review['votes_funny'],
                    'weighted_vote_score': review['weighted_vote_score'],
                })
            

quality_reviews_df = pd.DataFrame(all_reviews)
quality_reviews_df.head()

100%|██████████| 100/100 [03:22<00:00,  2.03s/it]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,I LOVEIT,1752504033,True,0,0,0.5
1,730,Counter-Strike: Global Offensive,kokotina\r\n,1752504004,False,0,0,0.5
2,730,Counter-Strike: Global Offensive,good,1752503928,True,1,0,0.523809552192687988
3,730,Counter-Strike: Global Offensive,Нзх,1752503292,True,0,0,0.5
4,730,Counter-Strike: Global Offensive,game rat hay toi rat thích\r\n,1752503244,True,0,0,0.5


In [13]:
# check in on counts and if filtering now
print("Total reviews:", len(quality_reviews_df))

has_word = quality_reviews_df['review'].str.contains(r'\b\w+\b')
is_long = quality_reviews_df.review.str.len().ge(10)
english_only = quality_reviews_df['review'].str.fullmatch(r"[A-Za-z0-9\s.,!?\"'’\-():;]+", na=False)

filtered_df = quality_reviews_df[has_word & is_long & english_only]
print("Filtered reviews (≥10 chars and has english words):", len(filtered_df))

Total reviews: 10139
Filtered reviews (≥10 chars and has english words): 6790


In [14]:
# Get the remaining games that still have at least 100 reviews
min_df = filtered_df
min_df.name.value_counts()

name
Lost Ark                   95
Z1 Battle Royale           87
NBA 2K20                   83
Half-Life 2: Lost Coast    83
Heroes & Generals          82
                           ..
VRChat                     36
World of Tanks Blitz       33
CyberCorp                  30
Street Warriors Online     16
Black Squad                 7
Name: count, Length: 100, dtype: int64

In [15]:
min_df[:100].to_csv("quality_reviews.csv")

Ok, now we have a reasonable amount of data with some context around the game, the review, and the voted up tag.  

We're going to generate a corpus of input and then split the full data into 3 sets- train, validate and test - in 80:10:10 batches

# Start Here for Clean Analysis


In [16]:
# Start here for clean
import pandas as pd
min_df = pd.read_csv("quality_reviews.csv")
min_df.head()

,Unnamed: 0,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,1,730,Counter-Strike: Global Offensive,kokotina\r\n,1752504004,False,0,0,0.5
1,5,730,Counter-Strike: Global Offensive,Type shit ahha W cheating,1752502923,True,0,0,0.5
2,9,730,Counter-Strike: Global Offensive,very good slot machine :),1752502042,True,0,0,0.5
3,10,1172470,Apex Legends,"Just like me, this game is gay, but I love bei...",1752419657,True,0,0,0.5
4,11,1172470,Apex Legends,Good Shit!,1751979827,True,0,0,0.5


In [17]:
# Clean the punctuation
import re

def cleaned(text):
    return re.sub(r'\W+', '_', text).lower()

In [18]:
# Add input Field
df = min_df.copy().reset_index(drop=True)
df['input'] = 'TEXT1: ' + df.review

# Convert the target to boolean ints
# Our target is currently saved as boolean true false - so let's convert to int
df['voted_up'] = df['voted_up'].astype(int)

display(df['input'].head())
display(df.voted_up.head())

0                                  TEXT1: kokotina\r\n
1                     TEXT1: Type shit ahha W cheating
2                     TEXT1: very good slot machine :)
3    TEXT1: Just like me, this game is gay, but I l...
4                                    TEXT1: Good Shit!
Name: input, dtype: object

0    0
1    1
2    1
3    1
4    1
Name: voted_up, dtype: int64

In [19]:
# Now let's get experience with datasets ( required for hugging face transformers )
from datasets import Dataset,DatasetDict

# Select the columns we want to keep for the dataset/prediction purposes
columns = ['input', 'voted_up']
df = df[columns].copy()

ds = Dataset.from_pandas(df)

In [20]:
ds

Dataset({
    features: ['input', 'voted_up'],
    num_rows: 100
})

In [21]:
# We're going to create todenizers using deberta
model_name = 'microsoft/deberta-v3-small'

!pip install -Uq transformers
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_name)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 79.6 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


The warning is only an issue because we're using noisy or multilingual data - the reviews, even when marked english, have all kinds of randomness.

Our options are to ignore (if we're ok with slighly less flexible outcomes) - instead of <unk> the characters would be byte level split.  
Or we can use something like..  
> from transformers import T5Tokenizer  
> tokenizer = T5Tokenizer.from_pretrained("t5-base", use_fast=False)  

In [22]:
# Test out the tokenizer with some basic text
tokz.tokenize("TEXT1: An Hello, I'm new at this and learning is fun!")

['▁TEXT',
 '1',
 ':',
 '▁An',
 '▁Hello',
 ',',
 '▁I',
 "'",
 'm',
 '▁new',
 '▁at',
 '▁this',
 '▁and',
 '▁learning',
 '▁is',
 '▁fun',
 '!']

In [23]:
# Vs some other text in the head of an earlier preview
tokz.tokenize("Tässä pelissä on intensiivistä väkivaltaa")

['▁T',
 'ä',
 's',
 's',
 'ä',
 '▁pe',
 'liss',
 'ä',
 '▁on',
 '▁in',
 't',
 'ensi',
 'ivist',
 'ä',
 '▁vä',
 'k',
 'ival',
 'ta',
 'a']

The tokenizer works fine on normal text but not fine on other text

In [24]:
# simple function to tokenize our
def tok_func(x): return tokz(x["input"])

In [25]:
# Parallel for every row in ds with map
tok_ds = ds.map(tok_func, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [26]:
row = tok_ds[0]
row['input'], row['input_ids']

('TEXT1: kokotina\r\n', [1, 54453, 435, 294, 10366, 4712, 24084, 2])

These ids are a list of vocab in the tokenizer with a unique int for every string

In [27]:
# See the int above
tokz.vocab['love']

10439

In [28]:
# Transformers needs a column called labels
tok_ds = tok_ds.rename_columns({'voted_up':'labels'})

In [29]:
tok_ds

Dataset({
    features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100
})

`tok_ds` is now ready for splitting

In [30]:
# Step 1: Train/Test Split (e.g., 80% train, 20% temp)
train_test = tok_ds.train_test_split(test_size=0.2, seed=42)

# Step 2: Split test portion into validation and test (e.g., 50/50 of the 20%)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

train_ds = train_test['train']
test_ds = val_test['train']
eval_ds = val_test['test']

dds = DatasetDict({
    'train': train_ds,
    'test': test_ds,
    'eval': eval_ds
})

In [31]:
dds

DatasetDict({
    train: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 80
    })
    test: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10
    })
    eval: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10
    })
})

In [32]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model_name = 'microsoft/deberta-v3-small'
tokz = AutoTokenizer.from_pretrained(model_name)

# Define metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Load model and enable checkpointing
from transformers import DebertaV2ForSequenceClassification

model = DebertaV2ForSequenceClassification.from_pretrained(model_name, num_labels=2)


# Define training arguments
args = TrainingArguments(
    output_dir='outputs',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    gradient_checkpointing=False,
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,   # Make sure you define these
    eval_dataset=eval_ds,
    tokenizer=tokz,
    compute_metrics=compute_metrics
)

trainer.train()


2025-07-14 20:42:37.756173: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752525757.952295      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752525758.011968      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_19/1352258326.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.715707,0.000000,0.000000
2,No log,0.608544,1.000000,1.000000
3,No log,0.584322,1.000000,1.000000


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=15, training_loss=0.6898005803426107, metrics={'train_runtime': 12.1152, 'train_samples_per_second': 19.81, 'train_steps_per_second': 1.238, 'total_flos': 16132620028608.0, 'train_loss': 0.6898005803426107, 'epoch': 3.0})

In [33]:
# Get raw logits
raw_preds = trainer.predict(eval_ds)

# Convert logits to class predictions (0 or 1)
preds = np.argmax(raw_preds.predictions, axis=1)

# Get ground truth labels
true_labels = raw_preds.label_ids

# Optionally inspect predictions and labels
print("Predictions:", preds[:10])
print("True Labels:", true_labels[:10])

# Compute accuracy
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(true_labels, preds)
print(f"\nAccuracy: {accuracy:.4f}")

# Optional: print precision, recall, F1
print("\nClassification Report:")
print(classification_report(true_labels, preds))

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Predictions: [1 1 1 1 1 1 1 1 1 1]
True Labels: [1 1 1 1 1 1 1 1 1 1]

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        10

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [34]:
# Get predictions from trainer
raw_preds = trainer.predict(eval_ds)
preds = np.argmax(raw_preds.predictions, axis=1)
true_labels = raw_preds.label_ids

# Convert dataset to pandas (only if it's not already)
df_eval = eval_ds.to_pandas()

# Add predictions and true labels to DataFrame
df_eval["predicted"] = preds
df_eval["label"] = true_labels

# Preview
display(df_eval)

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


,input,labels,input_ids,token_type_ids,attention_mask,predicted,label
0,"TEXT1: Fun and easy game. Easy after 300 hrs, ...",1,"[1, 54453, 435, 294, 6865, 263, 639, 522, 260,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1,1
1,TEXT1: If you are new or returning wait for th...,1,"[1, 54453, 435, 294, 369, 274, 281, 353, 289, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1,1
2,TEXT1: shoot plane plane go boom\r\n,1,"[1, 54453, 435, 294, 3841, 3853, 3853, 424, 10...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
3,"TEXT1: very fun, Pyro my beloved",1,"[1, 54453, 435, 294, 379, 785, 261, 53944, 312...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
4,TEXT1: try this game goatttttttttttttttttttttt...,1,"[1, 54453, 435, 294, 687, 291, 522, 13732, 111...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1,1
5,TEXT1: very good slot machine :),1,"[1, 54453, 435, 294, 379, 397, 4088, 1494, 877...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
6,TEXT1: Kinda fun to play but not really,1,"[1, 54453, 435, 294, 45616, 785, 264, 612, 304...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
7,TEXT1: for democracy,1,"[1, 54453, 435, 294, 270, 6053, 2]","[0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1]",1,1
8,TEXT1: When sex with pal? LoL,1,"[1, 54453, 435, 294, 486, 7454, 275, 15928, 30...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
9,TEXT1: had so much fun playing with my friends.,1,"[1, 54453, 435, 294, 330, 324, 400, 785, 1185,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
